In [0]:
from pyspark.sql.functions import *
import matplotlib.pyplot as plt

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [0]:

//===================//
#Load Clean Dataset
//==================//
df = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv("dbfs:/Volumes/workspace/default/uber_data/uber_trips_dataset_50k.csv")

# Remove invalid distance
df = df.filter(col("distance_km") > 0)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [0]:

//Dataset Overview//
print("="*60)
print("DATASET OVERVIEW")
print("="*60)

print("Rows :", df.count())
print("Columns :", len(df.columns))

display(df.limit(10))

DATASET OVERVIEW
Rows : 49997
Columns : 14


trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method,pickup_time,drop_time
1,8270,10683,San Francisco,37.17093115042939,-77.58647936979322,37.1736523397383,-77.61993433646886,2.97,10.71,Completed,Wallet,2023-01-01T00:00:00.000Z,2023-01-01T00:08:54.600Z
2,1860,44743,Boston,38.89812691856606,-108.58297695484846,38.93746379536704,-108.55872717079913,8.43,22.41,Completed,UPI,2023-01-01T00:01:00.000Z,2023-01-01T00:26:17.400Z
3,6390,75839,San Francisco,38.81457056869823,-89.94260270822319,38.82170236125605,-89.8964345566054,5.46,12.91,Completed,Cash,2023-01-01T00:02:00.000Z,2023-01-01T00:18:22.800Z
4,6191,22189,New York,37.29590568598714,-75.32884414927358,37.30137520518609,-75.31748783882725,6.61,15.7,Completed,Wallet,2023-01-01T00:03:00.000Z,2023-01-01T00:22:49.800Z
5,6734,61104,Seattle,38.97239533578873,-121.48291286204801,38.99208841336341,-121.46790441380276,10.5,19.15,Completed,Wallet,2023-01-01T00:04:00.000Z,2023-01-01T00:35:30.000Z
6,7265,84988,San Francisco,37.600550944815616,-106.71906865688572,37.55691580938672,-106.73095912437086,9.94,19.95,Completed,Card,2023-01-01T00:05:00.000Z,2023-01-01T00:34:49.200Z
7,1466,82933,San Francisco,40.28855964243779,-82.8082193858109,40.27883011229549,-82.84463502060092,12.22,25.89,Completed,UPI,2023-01-01T00:06:00.000Z,2023-01-01T00:42:39.600Z
8,5426,60182,Chicago,40.601188277022246,-78.39873323235332,40.569810860072174,-78.36818593976587,10.14,25.55,Completed,Cash,2023-01-01T00:07:00.000Z,2023-01-01T00:37:25.200Z
9,6578,15826,San Francisco,40.29624711069893,-87.57651655291244,40.284637585308246,-87.56478285126917,1.88,7.97,Completed,Cash,2023-01-01T00:08:00.000Z,2023-01-01T00:13:38.400Z
10,9322,63503,Boston,40.71263418106593,-78.49965570372784,40.67026654819631,-78.477475359972,3.7,12.21,No-Show,UPI,2023-01-01T00:09:00.000Z,2023-01-01T00:20:06.000Z


In [0]:

#Summary Statistics
display(df.describe())

summary,trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method
count,49997,49997,49997,49997,49997,49997,49997,49997,49997,49997,49997,49997
mean,24999.53917235034,5494.118947136828,55040.44966698002,null,38.998656921963935,-97.4866205660533,38.99876018979118,-97.48674627563362,7.008490709442576,15.976964017841079,null,null
stddev,14433.789861338271,2601.396679241189,25915.89357381104,null,1.1552184229710065,14.173476565158714,1.1556739222563828,14.173467145879444,2.946332438840855,6.2738257295296425,null,null
min,1,1000,10001,Boston,37.000009105082164,-121.99946522741315,36.953364823297214,-122.04712350229215,0.01,1.08,Cancelled,Card
max,50000,9998,99998,Seattle,40.99993677379934,-73.00196097873916,41.04753634317474,-72.96347649069838,19.41,50.67,No-Show,Wallet


In [0]:
#Trips by City

city_df = (
    df.groupBy("city")
      .count()
      .orderBy(col("count").desc())
)

display(city_df)

city,count
Boston,8454
San Francisco,8401
Chicago,8344
Los Angeles,8315
New York,8247
Seattle,8236


In [0]:
#Trip Status Analysis
status_df = (
    df.groupBy("status")
      .count()
)

display(status_df)

status,count
No-Show,2475
Cancelled,4984
Completed,42538


In [0]:
#Payment Method Analysis
payment_df = (
    df.groupBy("payment_method")
      .count()
)

display(payment_df)

payment_method,count
Wallet,12505
Card,12531
UPI,12548
Cash,12413


In [0]:
#Distance Distribution
display(
    df.select("distance_km")
)

distance_km
2.97
8.43
5.46
6.61
10.5
9.94
12.22
10.14
1.88
3.7


In [0]:
#Fare Distribution
display(
    df.select("fare_amount")
)

fare_amount
10.71
22.41
12.91
15.7
19.15
19.95
25.89
25.55
7.97
12.21


In [0]:
#10. Average Fare by City
from pyspark.sql.functions import avg, round

fare_city = (
    df.groupBy("city")
      .agg(round(avg("fare_amount"),2).alias("Average_Fare"))
      .orderBy(col("Average_Fare").desc())
)

display(fare_city)

city,Average_Fare
Seattle,16.06
New York,16.05
Boston,16.05
San Francisco,16.02
Chicago,15.9
Los Angeles,15.77


In [0]:
#11. Average Distance by City
distance_city = (
    df.groupBy("city")
      .agg(round(avg("distance_km"),2).alias("Average_Distance"))
      .orderBy(col("Average_Distance").desc())
)

display(distance_city)

city,Average_Distance
Seattle,7.07
San Francisco,7.03
Boston,7.02
New York,7.01
Chicago,6.96
Los Angeles,6.95


In [0]:
#Trips by Pickup Hour
from pyspark.sql.functions import hour

df = df.withColumn(
    "pickup_hour",
    hour("pickup_time")
)
hour_df = (
    df.groupBy("pickup_hour")
      .count()
      .orderBy("pickup_hour")
)

display(hour_df)

pickup_hour,count
0,2100
1,2100
2,2100
3,2100
4,2100
5,2100
6,2100
7,2100
8,2098
9,2100


In [0]:
#Trip Duration Analysis
from pyspark.sql.functions import unix_timestamp

df = df.withColumn(
    "trip_duration_min",
    (
        unix_timestamp("drop_time") -
        unix_timestamp("pickup_time")
    ) / 60
)

display(
    df.select(
        "trip_duration_min"
    )
)

trip_duration_min
8.9
25.283333333333335
16.366666666666667
19.816666666666666
31.5
29.816666666666666
36.65
30.416666666666668
5.633333333333334
11.1


In [0]:
display(
    df.select("trip_duration_min").describe()
)

summary,trip_duration_min
count,49997
mean,21.01876945950091
stddev,8.839003813014738
min,0.016666666666666666
max,58.21666666666667


In [0]:
#Correlation Between Distance and Fare
import builtins

corr = df.stat.corr("distance_km", "fare_amount")

print("Correlation :", builtins.round(corr, 3))

Correlation : 0.871


In [0]:
#Top 10 Highest Fare Trips
display(
    df.orderBy(
        col("fare_amount").desc()
    ).limit(10)
)

trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method,pickup_time,drop_time,pickup_hour,trip_duration_min
5875,7102,96068,Boston,39.324231086651736,-117.49451632559429,39.34864923027412,-117.4878607400919,19.41,50.67,No-Show,Wallet,2023-01-05T01:54:00.000Z,2023-01-05T02:52:13.800Z,1,58.21666666666667
41392,3597,81564,Seattle,37.409824037909495,-83.63593798376641,37.3611233171462,-83.633528229639,19.3,49.34,Cancelled,Cash,2023-01-29T17:51:00.000Z,2023-01-29T18:48:54.000Z,17,57.9
20298,7678,43787,New York,38.42802458777815,-103.73439095001154,38.45683219478123,-103.73993705577251,17.7,48.2,Completed,Cash,2023-01-15T02:17:00.000Z,2023-01-15T03:10:06.000Z,2,53.1
15667,6593,98176,New York,37.26949525877059,-89.74833928768717,37.27272111953051,-89.72502980777807,18.6,47.93,Completed,UPI,2023-01-11T21:06:00.000Z,2023-01-11T22:01:48.000Z,21,55.8
25837,2821,97257,Seattle,39.3832981002621,-113.56679876808946,39.34763931153562,-113.57436635255483,17.66,47.9,Completed,Wallet,2023-01-18T22:36:00.000Z,2023-01-18T23:28:58.800Z,22,52.96666666666667
39082,5042,86234,Seattle,38.19321676540905,-117.90795152377534,38.19070706766729,-117.93402460828362,17.72,47.35,Completed,UPI,2023-01-28T03:21:00.000Z,2023-01-28T04:14:09.600Z,3,53.15
4379,5159,63580,Los Angeles,38.33407249447775,-87.81086260580301,38.292445480629674,-87.80979741307972,18.08,46.43,Completed,UPI,2023-01-04T00:58:00.000Z,2023-01-04T01:52:14.400Z,0,54.233333333333334
4388,9975,29327,Seattle,39.79641252555092,-119.82808833894437,39.75346295156117,-119.81149928184803,17.36,44.27,Completed,UPI,2023-01-04T01:07:00.000Z,2023-01-04T01:59:04.800Z,1,52.06666666666667
29471,9165,18976,Chicago,38.542631648588255,-79.36909201997611,38.57128691140564,-79.37741737230634,16.26,43.56,Completed,UPI,2023-01-21T11:10:00.000Z,2023-01-21T11:58:46.800Z,11,48.766666666666666
38093,6984,18642,Boston,37.17296106290818,-89.79503299000302,37.16274688102981,-89.77713963246498,15.97,43.54,Completed,UPI,2023-01-27T10:52:00.000Z,2023-01-27T11:39:54.600Z,10,47.9


In [0]:
#Top 10 Longest Trips
display(
    df.orderBy(
        col("distance_km").desc()
    ).limit(10)
)

trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method,pickup_time,drop_time,pickup_hour,trip_duration_min
5875,7102,96068,Boston,39.324231086651736,-117.49451632559429,39.34864923027412,-117.4878607400919,19.41,50.67,No-Show,Wallet,2023-01-05T01:54:00.000Z,2023-01-05T02:52:13.800Z,1,58.21666666666667
41392,3597,81564,Seattle,37.409824037909495,-83.63593798376641,37.3611233171462,-83.633528229639,19.3,49.34,Cancelled,Cash,2023-01-29T17:51:00.000Z,2023-01-29T18:48:54.000Z,17,57.9
15667,6593,98176,New York,37.26949525877059,-89.74833928768717,37.27272111953051,-89.72502980777807,18.6,47.93,Completed,UPI,2023-01-11T21:06:00.000Z,2023-01-11T22:01:48.000Z,21,55.8
5195,4020,86968,Chicago,39.23312975884943,-118.20560446688407,39.25458147401182,-118.21888860443886,18.51,24.18,No-Show,Wallet,2023-01-04T14:34:00.000Z,2023-01-04T15:29:31.800Z,14,55.516666666666666
31396,5087,42876,Boston,40.98433807419432,-104.69172181080098,40.93999896304739,-104.67765206287359,18.46,41.87,Completed,Wallet,2023-01-22T19:15:00.000Z,2023-01-22T20:10:22.800Z,19,55.36666666666667
22904,2890,31248,Boston,38.793855872198364,-93.06314010605408,38.78161804842127,-93.03841913815607,18.42,24.22,Completed,UPI,2023-01-16T21:43:00.000Z,2023-01-16T22:38:15.600Z,21,55.25
27369,6417,31573,Seattle,40.254663349954455,-99.77147440349621,40.28021481530606,-99.74846045702297,18.33,37.45,Cancelled,UPI,2023-01-20T00:08:00.000Z,2023-01-20T01:02:59.400Z,0,54.983333333333334
36019,8171,82745,Seattle,37.723286947539954,-118.72437670544777,37.70578647790461,-118.73781784941583,18.12,40.84,Completed,Card,2023-01-26T00:18:00.000Z,2023-01-26T01:12:21.600Z,0,54.35
4379,5159,63580,Los Angeles,38.33407249447775,-87.81086260580301,38.292445480629674,-87.80979741307972,18.08,46.43,Completed,UPI,2023-01-04T00:58:00.000Z,2023-01-04T01:52:14.400Z,0,54.233333333333334
14635,2037,48106,Chicago,37.35043572837584,-117.82560002565094,37.36619348992176,-117.8430802279784,18.06,32.31,Completed,UPI,2023-01-11T03:54:00.000Z,2023-01-11T04:48:10.800Z,3,54.166666666666664


In [0]:
print("="*60)
print("BUSINESS INSIGHTS")
print("="*60)

print("• Seattle and San Francisco have strong trip activity.")
print("• Most trips are Completed.")
print("• Customers frequently use Wallet and UPI.")
print("• Fare increases as trip distance increases.")
print("• Peak pickup hours can be identified using hourly analysis.")
print("• Average fare varies across cities.")

BUSINESS INSIGHTS
• Seattle and San Francisco have strong trip activity.
• Most trips are Completed.
• Customers frequently use Wallet and UPI.
• Fare increases as trip distance increases.
• Peak pickup hours can be identified using hourly analysis.
• Average fare varies across cities.
